In [0]:
from pyspark.sql.functions import get_json_object

df_bronze = spark.table("gharchive.bronze_dev.events")


In [0]:
df_watchevents = df_bronze.filter("event_type = 'WatchEvent'") \
                          .select(get_json_object("payload", "$.action").alias("event_action"))
                        
df_watchevents.show(5, truncate=False)

In [0]:
df_watchevents = df_bronze.filter("event_type = 'WatchEvent'") \
                          .select(get_json_object("payload", "$.action").alias("event_action"))
                        
df_watchevents.show(5, truncate=False)

In [0]:
%sql
select distinct event_type, count(event_id) from gharchive.bronze_dev.events group by event_type

In [0]:
from pyspark.sql.functions import get_json_object, col

dates = ["2020-06-08", "2020-06-09", "2020-06-10", "2020-06-11",
         "2020-06-12", "2020-06-13", "2020-06-14"]

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gharchive.silver_dev")
spark.sql("CREATE SCHEMA IF NOT EXISTS gharchive.silver_prod")

In [0]:
%sql
drop table gharchive.silver_dev.issue_comment_events

In [0]:
for d in dates:
    df_watchevents = spark.table("gharchive.bronze_dev.events") \
                          .filter((col("event_type") == "WatchEvent") & (col("event_date") == d)) \
                          .select(
                              "event_id",
                              "actor_id",
                              "actor_login",
                              "repo_id",
                              "repo_name",
                              "org_id",
                              "created_at",
                              "event_date",
                              get_json_object("payload", "$.action").alias("event_action")
        )
    df_watchevents.write.format("delta").mode("overwrite").option("replaceWhere", f"event_date = '{d}'") \
        .partitionBy("event_date").saveAsTable("gharchive.silver_dev.watch_events")


In [0]:
%sql
select count(event_action) from gharchive.silver_dev.watch_events

In [0]:
for d in dates:
    df_pushevents = spark.table("gharchive.bronze_dev.events") \
                          .filter((col("event_type") == "PushEvent") & (col("event_date") == d)) \
                          .select(
                              "event_id",
                              "actor_id",
                              "actor_login",
                              "repo_id",
                              "repo_name",
                              "org_id",
                              "created_at",
                              "event_date",
                              get_json_object("payload", "$.size").alias("push_size")
        )
    df_pushevents.write.format("delta").mode("overwrite").option("replaceWhere", f"event_date = '{d}'") \
        .partitionBy("event_date").saveAsTable("gharchive.silver_dev.push_events")

In [0]:
for d in dates:
    df_pr = spark.table("gharchive.bronze_dev.events") \
        .filter((col("event_type") == "PullRequestEvent") & (col("event_date") == d)) \
        .select(
                              "actor_id",
                              "actor_login",
                              "repo_id",
                              "repo_name",
                              "org_id",
                              "created_at","event_date",
            get_json_object("payload", "$.action").alias("event_action"),
            get_json_object("payload", "$.number").cast("int").alias("pr_number"),
            get_json_object("payload", "$.pull_request.merged").cast("boolean").alias("pr_merged")
        )
    df_pr.write.format("delta").mode("overwrite").option("replaceWhere", f"event_date = '{d}'") \
        .partitionBy("event_date").saveAsTable("gharchive.silver_dev.pull_request_events")

In [0]:
for d in dates:
    df_ie = spark.table("gharchive.bronze_dev.events") \
        .filter((col("event_type") == "IssuesEvent") & (col("event_date") == d)) \
        .select(
            
                              "actor_id",
                              "actor_login",
                              "repo_id",
                              "repo_name",
                              "org_id",
                              "created_at","event_date",
            get_json_object("payload", "$.action").alias("event_action"),
            get_json_object("payload", "$.issue.number").cast("int").alias("issue_number")
        )
    df_ie.write.format("delta").mode("overwrite").option("replaceWhere", f"event_date = '{d}'") \
        .partitionBy("event_date").saveAsTable("gharchive.silver_dev.issues_events")

In [0]:
for d in dates:
    df_ice = spark.table("gharchive.bronze_dev.events") \
        .filter((col("event_type") == "IssueCommentEvent") & (col("event_date") == d)) \
        .select(
            
                              "actor_id",
                              "actor_login",
                              "repo_id",
                              "repo_name",
                              "org_id",
                              "created_at",
                              "event_date",
            get_json_object("payload", "$.action").alias("event_action"),
            get_json_object("payload", "$.issue.number").cast("int").alias("issue_number")
        )
    df_ice.write.format("delta").mode("overwrite").option("replaceWhere", f"event_date = '{d}'") \
        .partitionBy("event_date").saveAsTable("gharchive.silver_dev.issue_comment_events")

In [0]:
for table in ["push_events", "pull_request_events", "issues_events", "issue_comment_events", "watch_events"]:
    count = spark.table(f"gharchive.silver_dev.{table}").count()
    print(f"{table}: {count}")

In [0]:
spark.table("gharchive.silver_dev.watch_events").select("event_action").distinct().show()